In [85]:
# import packages
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import layers

In [86]:
# create fully connected layer using keras for accomodating msnit data as input
model = keras.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(10)
])

In [87]:
# validate neural network layers
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_2 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,386 (427.29 KB)

 Trainable params: 109,386 (427.29 KB)

 Non-trainable params: 0 (0.00 B)

In [88]:
# split data in to train and test variables for inference
(x_train, y_train),(x_test, y_test) = keras.datasets.mnist.load_data()

# scale the gray scale pixels to range of 0 to 1
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print(x_train.shape)
print(x_test.shape)

(60000, 28, 28)
(10000, 28, 28)


In [89]:
# set training parameters
model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

In [90]:
# train the model with given inputs
history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)

Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.9167 - loss: 0.2927 - val_accuracy: 0.9605 - val_loss: 0.1342
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9644 - loss: 0.1187 - val_accuracy: 0.9712 - val_loss: 0.0967
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9745 - loss: 0.0830 - val_accuracy: 0.9748 - val_loss: 0.0840
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.9805 - loss: 0.0632 - val_accuracy: 0.9798 - val_loss: 0.0724
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9846 - loss: 0.0492 - val_accuracy: 0.9802 - val_loss: 0.0696


In [91]:
# test the data with testing data with trained weights
original_test_loss, original_test_accuracy = model.evaluate(
    x_test,
    y_test
)

print("Test accuracy:", original_test_accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9788 - loss: 0.0746
Test accuracy: 0.9787999987602234


In [92]:
pruned_weights_mask = []
def fine_tune_layers():
  #load all weights inside a list
  for layer in model.layers:
    if isinstance(layer, keras.layers.Dense):
        weights, biases = layer.get_weights()
        threshold = np.percentile(np.abs(weights), 30)
        mask = np.abs(weights) < threshold
        pruned_weights_mask.append(mask)
        weights[mask] = 0
        layer.set_weights([weights, biases])

In [93]:
fine_tune_layers()

In [99]:
for epoch in range(5):
    print(f"Epoch {epoch + 1}/3")

    model.fit(
        x_train,
        y_train,
        epochs=1,
        batch_size=64,
        validation_split=0.1
    )

    dense_index = 0

    for layer in model.layers:
        if isinstance(layer, keras.layers.Dense):
            weights, bias = layer.get_weights()

            weights[pruned_weights_mask[dense_index]] = 0

            layer.set_weights([weights, bias])

            dense_index += 1

Epoch 1/3
844/844 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9939 - loss: 0.0185 - val_accuracy: 0.9795 - val_loss: 0.0875
Epoch 2/3
844/844 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.9945 - loss: 0.0164 - val_accuracy: 0.9805 - val_loss: 0.0800
Epoch 3/3
844/844 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9949 - loss: 0.0153 - val_accuracy: 0.9798 - val_loss: 0.0816
Epoch 4/3
844/844 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9952 - loss: 0.0141 - val_accuracy: 0.9782 - val_loss: 0.0972
Epoch 5/3
844/844 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9960 - loss: 0.0115 - val_accuracy: 0.9813 - val_loss: 0.0868


In [100]:
pruned_test_loss, pruned_test_accuracy = model.evaluate(x_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9805 - loss: 0.0823


In [101]:
for index, layer in enumerate(model.layers):
    if isinstance(layer, keras.layers.Dense):
        weights, bias = layer.get_weights()

        zeros_count = np.sum(weights == 0)
        total_count = weights.size

        print(f"layer {index}:")
        print("zero weights:", zeros_count)
        print("sparsity:", zeros_count / total_count)

layer 1:
zero weights: 30106
sparsity: 0.30000398596938777
layer 2:
zero weights: 2458
sparsity: 0.300048828125
layer 3:
zero weights: 192
sparsity: 0.3


In [102]:
def round_fun(x):
  return round(x*100, 4)

In [103]:
print("1. Accuracy before pruning:", round_fun(original_test_accuracy), "%")
print("2. Accuracy after pruning:", round_fun(pruned_test_accuracy), "%")
print("Removed 30% of the weights while gradually increasing the accuracy.")


1. Accuracy before pruning: 97.88 %
2. Accuracy after pruning: 98.05 %
Removed 30% of the weights while gradually increasing the accuracy.
